# 01_07_drp_patch_from_existing_table

Тетрадка для внесения последних форматных правок **поверх уже загруженной DRP-таблицы** (без пересчета из Озера).

Что делает:
- создает новую целевую таблицу/вью на основе существующей DRP-таблицы;
- добавляет `tariff_short` по актуальным правилам сегментации;
- добавляет `filial_rf_norm` и `ssp_ocrm_norm`;
- добавляет numeric-колонки с округлением до 2 знаков для денежных полей;
- выводит быстрые проверки после публикации.

Запуск: сверху вниз, по одной ячейке.

In [ ]:
from getpass import getpass

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Конфиг источника/цели
source_schema = 'sbx_da'
source_table = 'tmp_shestopalov_acq_datamart_q1_v2'

target_schema = 'sbx_da'
target_name = 'tmp_shestopalov_acq_datamart_q1_v3'

publish_mode = 'table'   # 'table' или 'view'
drop_target_if_exists = True

source_fq = f'{source_schema}.{source_table}'
target_fq = f'{target_schema}.{target_name}'

print('source =', source_fq)
print('target =', target_fq)
print('publish_mode =', publish_mode)

In [ ]:
# Подключение к DRP
drp_user = input('DRP user: ').strip()
drp_password = getpass('DRP password: ')

drp_conn = connect(
    to='DRP',
    user_params={
        'user_name': drp_user,
        'password': drp_password,
    }
)

print('DRP connection initialized')

In [ ]:
# Публикация target (table/view) с последними правками
transform_sql = f"""
select
    t.*,

    -- Короткий тариф по правилам 3.21
    case
        when lower(coalesce(t.tariff_name, '')) like '%акт%' then 'По актам'
        when lower(coalesce(t.tariff_name, '')) like '%акцион%'
          or lower(coalesce(t.tariff_name, '')) like '%меню%'
          or lower(coalesce(t.tariff_name, '')) like '%сезон%' then 'Акционный'
        when lower(coalesce(t.tariff_name, '')) like '%индив%' then 'Индивидуальный'
        when lower(coalesce(t.tariff_name, '')) like '%стандарт%' then 'Стандарт'
        else 'Не сегментирован'
    end as tariff_short,

    -- Нормализованный филиал
    case
        when nullif(btrim(cast(t.filial_rf as text)), '') is null then null
        else replace(
            initcap(
                regexp_replace(
                    coalesce(substring(lower(cast(t.filial_rf as text)) from '^(.*?рф)'), lower(cast(t.filial_rf as text))),
                    '\\s+', ' ', 'g'
                )
            ),
            'Рф', 'РФ'
        )
    end as filial_rf_norm,

    -- Нормализованный сегмент OCRM
    case
        when lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дкб%' then 'ДКБ'
        when replace(lower(coalesce(cast(t.ssp_ocrm as text), '')), ' ', '') like 'дмсб(ми%'
          or lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дммб%' then 'ДМ'
        when lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дмсб%'
          or lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дсб%' then 'ДМСБ'
        when lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дм%' then 'ДМ'
        else null
    end as ssp_ocrm_norm,

    -- Денежные поля в numeric(2)
    round(cast(nullif(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.'), '') as numeric), 2) as trx_sum_num,
    round(cast(nullif(replace(replace(btrim(cast(t.commission_from_ops as text)), ' ', ''), ',', '.'), '') as numeric), 2) as commission_from_ops_num,
    round(cast(nullif(replace(replace(btrim(cast(t.commission_monthly as text)), ' ', ''), ',', '.'), '') as numeric), 2) as commission_monthly_num,
    round(cast(nullif(replace(replace(btrim(cast(t.commission_total as text)), ' ', ''), ',', '.'), '') as numeric), 2) as commission_total_num,
    round(cast(nullif(replace(replace(btrim(cast(t.int_component as text)), ' ', ''), ',', '.'), '') as numeric), 2) as int_component_num,
    round(cast(nullif(replace(replace(btrim(cast(t.chod as text)), ' ', ''), ',', '.'), '') as numeric), 2) as chod_num,
    round(cast(nullif(replace(replace(btrim(cast(t.fin_result as text)), ' ', ''), ',', '.'), '') as numeric), 2) as fin_result_num

from {source_fq} t
"""

with drp_conn:
    src_cnt_df = drp_conn.fetch(f"select count(*) as row_cnt from {source_fq}")

src_cnt = int(pd.to_numeric(src_cnt_df.iloc[0, 0], errors='coerce')) if src_cnt_df is not None and len(src_cnt_df) else 0
print('source rows =', src_cnt)

with drp_conn:
    if publish_mode == 'table':
        if drop_target_if_exists:
            drp_conn.execute(f'DROP TABLE IF EXISTS {target_fq}')
        drp_conn.execute(f'CREATE TABLE {target_fq} AS {transform_sql}')
    elif publish_mode == 'view':
        if drop_target_if_exists:
            drp_conn.execute(f'DROP VIEW IF EXISTS {target_fq}')
        drp_conn.execute(f'CREATE VIEW {target_fq} AS {transform_sql}')
    else:
        raise ValueError("publish_mode должен быть 'table' или 'view'")

print('Published:', target_fq)

In [ ]:
# Проверка результата
with drp_conn:
    src_cnt_df = drp_conn.fetch(f"select count(*) as row_cnt from {source_fq}")
    tgt_cnt_df = drp_conn.fetch(f"select count(*) as row_cnt from {target_fq}")

    segment_df = drp_conn.fetch(f"""
        select tariff_short, count(*) as rows
        from {target_fq}
        group by tariff_short
        order by rows desc
    """)

    sample_df = drp_conn.fetch(f"""
        select
            report_month,
            inn,
            tariff_name,
            tariff_short,
            filial_rf,
            filial_rf_norm,
            ssp_ocrm,
            ssp_ocrm_norm,
            trx_sum,
            trx_sum_num,
            commission_total,
            commission_total_num,
            fin_result,
            fin_result_num
        from {target_fq}
        limit 20
    """)

src_cnt = int(pd.to_numeric(src_cnt_df.iloc[0, 0], errors='coerce')) if src_cnt_df is not None and len(src_cnt_df) else 0
tgt_cnt = int(pd.to_numeric(tgt_cnt_df.iloc[0, 0], errors='coerce')) if tgt_cnt_df is not None and len(tgt_cnt_df) else 0

print('source rows =', src_cnt)
print('target rows =', tgt_cnt)
print('delta rows =', tgt_cnt - src_cnt)

print('\nTariff short distribution:')
display(segment_df)

print('\nTarget sample:')
display(sample_df)

## Что дальше

1. Если `target rows` совпадает с `source rows`, можно создавать/переключать Dataset в Superset на `target_fq`.
2. Для KPI и графиков используйте `*_num` колонки (например `fin_result_num`, `commission_total_num`, `trx_sum_num`).
3. Если хотите оставить исходную таблицу как backup, ничего в ней не меняется — правки только в новой целевой таблице/вью.